# 02 · EDA y Data Quality Report — Grupo 17
**Paso 3 del Sprint 2.** Responde preguntas de decisión, no acumula gráficos:

| # | Pregunta | Decisión que habilita |
|---|---|---|
| Q1 | ¿Cómo queda la clase × origen con el mapeo propuesto? | Diccionario de etiquetas v1 (Paso 4) |
| Q2 | ¿Hay duplicados dentro y entre fuentes, o textos con dos clases? | Regla de deduplicación |
| Q3 | ¿Qué idioma tiene cada fuente? | Filtro de idioma para DistilBERT |
| Q4 | ¿Enron y Ling vienen sin URLs por preprocesamiento? | Qué corpus usar desde CSV y cuál desde crudo |
| Q5 | ¿Cuánto separa el tiempo a las clases? | Declaración del confusor temporal |
| Q6 | ¿Un modelo puede adivinar la FUENTE? ¿Y la clase con solo metadatos? | Tamaño del riesgo de artefactos |
| Q7 | PhiUSIIL: ¿`FILENAME` o `URLSimilarityIndex` filtran la etiqueta? | Qué columnas se descartan |

**Salidas:** `data/interim/emails_v0.parquet` (solo texto **defangeado**), `data/eda/revision_manual_pot.csv`
(50 correos para revisión humana), `data/eda/dqr_v1.json` y el `RESUMEN`.
Requiere en Drive: `src/ioc_utils.py` v1.2 y `src/data_prep.py` v0.2.

In [1]:
# Celda 1: Entorno, semillas y módulos
import os, sys, json, re, mailbox, zipfile, random, importlib.util, time
from pathlib import Path
from collections import Counter
import numpy as np, pandas as pd
SEED = 17; random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)
def detectar_entorno():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"): return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "local"
ENV = detectar_entorno()
if ENV == "colab":
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/PI_G17")
elif ENV == "kaggle":
    BASE = Path("/kaggle/working/PI_G17")
else:
    BASE = Path(os.environ.get("PI_G17_BASE", Path.cwd() / "PI_G17"))
V1, COMP = BASE / "data/raw/g17-phishing-raw-v1", BASE / "data/raw/g17-phishing-complemento-v1"
INTERIM, EDA = BASE / "data/interim", BASE / "data/eda"
for p in (INTERIM, EDA): p.mkdir(parents=True, exist_ok=True)
def cargar(nombre, ruta):
    spec = importlib.util.spec_from_file_location(nombre, ruta); m = importlib.util.module_from_spec(spec)
    sys.modules[nombre] = m; spec.loader.exec_module(m); return m
ioc = cargar("ioc_utils", BASE / "src/ioc_utils.py"); dp = cargar("data_prep", BASE / "src/data_prep.py")
assert "v1.2" in ioc.__doc__ and hasattr(dp, "texto_plano"), "Faltan ioc_utils v1.2 o data_prep v0.2 en src/"
print(ENV, BASE)

Mounted at /content/drive
colab /content/drive/MyDrive/PI_G17


In [2]:
# Celda 2: Tabla unificada (mapeo PROPUESTO, se valida en Q1)
# Kaggle CEAS/Enron/Ling/SpamAssassin: 0 -> legitimo, 1 -> spam. Nazario/Nigerian/pot/Nazario anual -> phishing.
MAPEO = {"CEAS_08": {0: "legitimo", 1: "spam"}, "Enron": {0: "legitimo", 1: "spam"},
         "Ling": {0: "legitimo", 1: "spam"}, "SpamAssasin": {0: "legitimo", 1: "spam"},
         "Nazario": {1: "phishing"}, "Nigerian_Fraud": {1: "phishing"}}
def leer_csv(p):
    for enc in ("utf-8", "latin-1"):
        try: return pd.read_csv(p, encoding=enc, low_memory=False)
        except UnicodeDecodeError: pass
filas, errores = [], Counter()
t0 = time.time()
for s, mapa in MAPEO.items():
    df = leer_csv(next(V1.rglob(f"{s}.csv")))
    fechas = df["date"] if "date" in df.columns else pd.Series([None] * len(df))
    for subj, body, lab, fecha in zip(df["subject"].fillna(""), df["body"].fillna(""), df["label"], fechas):
        filas.append({"fuente": f"kaggle/{s}", "grupo": s, "etiqueta_original": int(lab),
                      "clase": mapa.get(int(lab), "SIN_MAPEO"), "subject": str(subj), "body": str(body),
                      "anio": dp.anio_de(fecha), "de_crudo": False})
for zp in sorted((COMP / "phishing_pot").glob("*.zip")):
    z = zipfile.ZipFile(zp)
    for n in z.namelist():
        if n.lower().endswith(".eml"):
            try:
                d = dp.texto_de_email(z.read(n))
                filas.append({"fuente": "phishing_pot", "grupo": "phishing_pot", "etiqueta_original": 1, "clase": "phishing",
                              "subject": d["subject"], "body": d["body"], "anio": dp.anio_de(d["date"]), "de_crudo": True})
            except Exception: errores["phishing_pot"] += 1
for f in sorted((COMP / "nazario_anual").glob("phishing-*")):
    for msg in mailbox.mbox(str(f)):
        try:
            d = dp.texto_de_email(msg)
            filas.append({"fuente": f"nazario/{f.name}", "grupo": "nazario_anual", "etiqueta_original": 1, "clase": "phishing",
                          "subject": d["subject"], "body": d["body"], "anio": dp.anio_de(d["date"]), "de_crudo": True})
        except Exception: errores[f.name] += 1
E = pd.DataFrame(filas); del filas
texto = E["subject"] + "\n" + E["body"]
E["plano"] = texto.map(dp.texto_plano)
E["hash"] = E["plano"].str.lower().map(lambda s: __import__("hashlib").sha1(s.encode("utf-8", "replace")).hexdigest())
E["n_urls"] = texto.map(lambda t: len(ioc.urls_para_features(t)))
E["sin_url"] = E["n_urls"] == 0
E["es_html"] = E["body"].str.contains(r"(?i)<(?:html|body|a |div|table)", regex=True)
E["len_plano"] = E["plano"].str.len()
print(f"{len(E)} correos en {time.time()-t0:.0f}s | errores de parseo: {dict(errores)}")
assert (E["clase"] != "SIN_MAPEO").all()

94566 correos en 130s | errores de parseo: {}


In [3]:
# Celda 3: Q1 clase × grupo  y  Q2 duplicados
q1 = pd.crosstab(E["grupo"], E["clase"], margins=True)
display(q1)
dup_intra = E.duplicated(["grupo", "hash"]).groupby(E["grupo"]).sum().astype(int).to_dict()
primer_grupo = E.groupby("hash")["grupo"].nunique()
cruce = E[E["hash"].isin(primer_grupo[primer_grupo > 1].index)]
pares_cruce = Counter(tuple(sorted(g)) for g in cruce.groupby("hash")["grupo"].unique())
clases_por_hash = E.groupby("hash")["clase"].nunique()
conflictos = E[E["hash"].isin(clases_por_hash[clases_por_hash > 1].index)]
vacios = int((E["len_plano"] < 20).sum())
q2 = {"duplicados_intra_grupo": dup_intra,
      "hashes_en_varios_grupos": int((primer_grupo > 1).sum()),
      "pares_de_grupos_con_cruce": {" + ".join(k): v for k, v in pares_cruce.most_common(8)},
      "hashes_con_clases_en_conflicto": int((clases_por_hash > 1).sum()),
      "correos_en_conflicto_por_grupo": conflictos["grupo"].value_counts().to_dict(),
      "textos_casi_vacios_<20_chars": vacios}
q2

clase,legitimo,phishing,spam,All
grupo,,,,
CEAS_08,17312,0,21842,39154
Enron,15791,0,13976,29767
Ling,2401,0,458,2859
Nazario,0,1565,0,1565
Nigerian_Fraud,0,3332,0,3332
SpamAssasin,4091,0,1718,5809
nazario_anual,0,3466,0,3466
phishing_pot,0,8614,0,8614
All,39595,16977,37994,94566


{'duplicados_intra_grupo': {'CEAS_08': 191,
  'Enron': 344,
  'Ling': 0,
  'Nazario': 13,
  'Nigerian_Fraud': 41,
  'SpamAssasin': 7,
  'nazario_anual': 340,
  'phishing_pot': 1766},
 'hashes_en_varios_grupos': 441,
 'pares_de_grupos_con_cruce': {'Nazario + nazario_anual': 439,
  'Enron + SpamAssasin': 2},
 'hashes_con_clases_en_conflicto': 0,
 'correos_en_conflicto_por_grupo': {},
 'textos_casi_vacios_<20_chars': 39}

In [4]:
# Celda 4: Q3 idioma
# Completo en las fuentes crudas nuevas (pot, Nazario anual); muestra de 1.500 por grupo en Kaggle.
%pip install -q langdetect
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0
def idioma(s):
    s = s[:1000]
    if len(s) < 40: return "corto"
    try: return detect(s)
    except Exception: return "desconocido"
t0 = time.time()
E["idioma"] = None
m_crudo = E["de_crudo"]
E.loc[m_crudo, "idioma"] = E.loc[m_crudo, "plano"].map(idioma)
muestra_k = E[~m_crudo].sample(frac=1, random_state=SEED).groupby("grupo").head(1500)   # estratificado, estable entre versiones de pandas
idioma_k = muestra_k["plano"].map(idioma)
tab = pd.concat([E.loc[m_crudo, ["grupo", "idioma"]], pd.DataFrame({"grupo": muestra_k["grupo"], "idioma": idioma_k})])
q3 = (tab.groupby("grupo")["idioma"].value_counts(normalize=True).mul(100).round(1)
         .groupby(level=0).head(4).unstack(fill_value=0))
print(f"{time.time()-t0:.0f}s"); q3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 12.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
89s


idioma,en,ca,fr,hr,es,corto,de,pt,tr
grupo,,,,,,,,,
CEAS_08,99.6,0.1,0.1,0.1,0.0,0.0,0.0,0.0,0.0
Enron,97.6,0.5,0.0,0.0,0.5,0.5,0.0,0.0,0.0
Ling,98.1,0.0,1.0,0.0,0.2,0.0,0.4,0.0,0.0
Nazario,95.8,0.0,0.5,0.0,0.5,0.0,0.0,0.7,0.0
Nigerian_Fraud,98.7,0.0,1.3,0.0,0.0,0.0,0.1,0.0,0.0
SpamAssasin,99.3,0.0,0.1,0.0,0.0,0.0,0.2,0.0,0.3
nazario_anual,94.0,0.0,0.0,0.0,1.0,1.4,0.0,0.8,0.0
phishing_pot,51.1,0.0,5.4,0.0,0.0,0.0,21.3,13.8,0.0


In [5]:
# Celda 5: Q4 ¿URLs eliminadas por preprocesamiento?  Q5 tiempo
patrones = {"http": r"(?i)http", "www": r"(?i)\bwww\b", "href": r"(?i)href", "dominio_com": r"(?i)\b\w+\s?\.\s?com\b",
            "token_url_sustituto": r"(?i)\b(?:url|link|hyperlink)\b"}
q4 = {g: {k: round(100 * float(E.loc[E.grupo == g, "body"].str.contains(p, regex=True).mean()), 1) for k, p in patrones.items()}
      for g in sorted(E.grupo.unique())}
q4 = pd.DataFrame(q4).T; q4["pct_con_url_valida"] = E.groupby("grupo")["sin_url"].apply(lambda s: round(100 * (1 - s.mean()), 1))
display(q4)
q5 = E.groupby(["clase", "grupo"])["anio"].describe(percentiles=[.5])[["count", "min", "50%", "max"]]
q5["pct_con_anio"] = (E.groupby(["clase", "grupo"])["anio"].apply(lambda s: s.notna().mean() * 100)).round(1)
q5

,http,www,href,dominio_com,token_url_sustituto,pct_con_url_valida
CEAS_08,67.1,26.6,0.1,65.8,13.7,67.5
Enron,18.8,11.5,0.5,27.4,5.8,0.0
Ling,38.8,31.2,0.2,17.2,8.1,0.0
Nazario,13.0,8.6,0.1,21.9,18.0,16.2
Nigerian_Fraud,33.3,22.7,0.1,50.2,3.3,36.7
SpamAssasin,79.0,50.6,0.5,63.4,19.7,79.2
nazario_anual,94.5,50.3,88.6,69.8,47.0,92.1
phishing_pot,96.8,39.2,91.6,77.7,35.4,92.9


count     min     50%     max  pct_con_anio
clase    grupo                                                        
legitimo CEAS_08         17312.0  2005.0  2008.0  2008.0         100.0
         Enron               0.0     NaN     NaN     NaN           0.0
         Ling                0.0     NaN     NaN     NaN           0.0
         SpamAssasin      4091.0  2002.0  2002.0  2028.0         100.0
phishing Nazario          1563.0  2015.0  2018.0  2022.0          99.9
         Nigerian_Fraud   2836.0  1998.0  2006.0  2015.0          85.1
         nazario_anual    3400.0  2015.0  2020.0  2025.0          98.1
         phishing_pot     8609.0  2019.0  2024.0  2026.0          99.9
spam     CEAS_08         21816.0  1998.0  2008.0  2020.0          99.9
         Enron               0.0     NaN     NaN     NaN           0.0
         Ling                0.0     NaN     NaN     NaN           0.0
         SpamAssasin      1632.0  1997.0  2002.0  2020.0          95.0

In [6]:
# Celda 6: Q6 pruebas de artefacto (muestra estratificada)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
m = E[E["len_plano"] >= 20]
S = m.sample(frac=1, random_state=SEED).groupby("grupo").head(2000)
def f1_texto(X, y):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=50000, sublinear_tf=True)
    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(vec.fit_transform(Xtr), ytr)
    return round(f1_score(yte, clf.predict(vec.transform(Xte)), average="macro"), 3)
def f1_meta(D, y):
    X = np.c_[np.log1p(D["len_plano"]), D["n_urls"].clip(0, 50), D["sin_url"], D["es_html"], D["subject"].str.len()]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
    from sklearn.ensemble import HistGradientBoostingClassifier
    clf = HistGradientBoostingClassifier(random_state=SEED, class_weight="balanced").fit(Xtr, ytr)
    return round(f1_score(yte, clf.predict(Xte), average="macro"), 3)
ph = S[S.clase == "phishing"]
q6 = {"n_muestra": int(len(S)),
      "f1_predecir_GRUPO_con_texto": f1_texto(S["plano"], S["grupo"]),
      "f1_predecir_GRUPO_de_phishing_con_texto": f1_texto(ph["plano"], ph["grupo"]),
      "f1_predecir_CLASE_solo_metadatos": f1_meta(S, S["clase"]),
      "f1_predecir_CLASE_con_texto (referencia)": f1_texto(S["plano"], S["clase"])}
q6

{'n_muestra': 15565,
 'f1_predecir_GRUPO_con_texto': 0.811,
 'f1_predecir_GRUPO_de_phishing_con_texto': 0.715,
 'f1_predecir_CLASE_solo_metadatos': 0.631,
 'f1_predecir_CLASE_con_texto (referencia)': 0.959}

In [7]:
# Celda 7: Q7 PhiUSIIL
P = pd.read_csv(next(V1.rglob("phiusiil_original.csv")), low_memory=False)
P["y_phish"] = 1 - P["label"]                                  # convención del proyecto: 1 = phishing
fn_pref = P["FILENAME"].astype(str).str.extract(r"^([A-Za-z_]+)")[0]
q7 = {"filas": int(len(P)), "urls_duplicadas": int(P["URL"].duplicated().sum()),
      "filename_unicos": int(P["FILENAME"].nunique()),
      "filename_prefijo_vs_etiqueta": pd.crosstab(fn_pref.fillna("?"), P["y_phish"]).head(6).to_dict(),
      "pct_https_por_clase": P.groupby("y_phish")["IsHTTPS"].mean().mul(100).round(1).to_dict(),
      "pct_URLSimilarityIndex_100_por_clase": P.groupby("y_phish")["URLSimilarityIndex"].apply(lambda s: round(100 * (s == 100).mean(), 1)).to_dict(),
      "pct_url_con_esquema_por_clase": P.groupby("y_phish")["URL"].apply(lambda s: round(100 * s.str.match(r"(?i)https?://").mean(), 1)).to_dict(),
      "pct_url_con_www_por_clase": P.groupby("y_phish")["URL"].apply(lambda s: round(100 * s.str.contains(r"(?i)//www\.").mean(), 1)).to_dict()}
q7

{'filas': 235795,
 'urls_duplicadas': 425,
 'filename_unicos': 235795,
 'filename_prefijo_vs_etiqueta': {0: {'?': 134850,
   'm': 0,
   'mw': 0,
   'op': 0,
   'oph': 0},
  1: {'?': 55211, 'm': 381, 'mw': 36286, 'op': 589, 'oph': 8478}},
 'pct_https_por_clase': {0: 100.0, 1: 49.2},
 'pct_URLSimilarityIndex_100_por_clase': {0: 100.0, 1: 0.8},
 'pct_url_con_esquema_por_clase': {0: 100.0, 1: 100.0},
 'pct_url_con_www_por_clase': {0: 100.0, 1: 41.4}}

In [8]:
# Celda 8: Artefactos derivados (siempre defangeados)
E["texto_defang"] = (E["subject"] + "\n" + E["body"]).map(ioc.defang_texto)
E["urls_defang"] = (E["subject"] + "\n" + E["body"]).map(lambda t: ioc.extraer_iocs(t)["urls"])
vivos = int(E["texto_defang"].map(lambda t: bool(ioc.iocs_vivos(t))).sum())
assert vivos == 0, f"{vivos} textos con IOC vivo: no se persiste nada"
cols = ["fuente", "grupo", "etiqueta_original", "clase", "anio", "idioma", "hash", "n_urls", "sin_url",
        "es_html", "len_plano", "texto_defang", "urls_defang"]
E[cols].to_parquet(INTERIM / "emails_v0.parquet", index=False)
# 50 correos de phishing_pot en inglés para revisión manual del ruido de etiquetas
rev = E[(E.grupo == "phishing_pot") & (E.idioma == "en")].sample(50, random_state=SEED)
rev = rev.assign(texto=rev["texto_defang"].map(lambda t: ioc.defang_texto(dp.texto_plano(t))[:1500]),
                 veredicto_revisor="", comentario="")[["hash", "anio", "texto", "veredicto_revisor", "comentario"]]
assert not rev["texto"].map(ioc.iocs_vivos).map(bool).any()
rev.to_csv(EDA / "revision_manual_pot.csv", index=False)
print("Guardado:", INTERIM / "emails_v0.parquet", "| revisión:", EDA / "revision_manual_pot.csv", "| IOC vivos:", vivos)

Guardado: /content/drive/MyDrive/PI_G17/data/interim/emails_v0.parquet | revisión: /content/drive/MyDrive/PI_G17/data/eda/revision_manual_pot.csv | IOC vivos: 0


In [9]:
# Celda 9: DQR y RESUMEN
dqr = {"entorno": ENV, "n_correos": int(len(E)), "errores_parseo": dict(errores),
       "Q1_clase_x_grupo": q1.to_dict(), "Q2_duplicados": q2,
       "Q3_idioma_pct": q3.to_dict(orient="index"), "Q4_patrones_url_pct": q4.to_dict(orient="index"),
       "Q5_anio": q5.round(0).reset_index().astype(str).to_dict(orient="records"),
       "Q6_artefactos_f1": q6, "Q7_phiusiil": q7, "ioc_vivos_en_derivados": vivos}
(EDA / "dqr_v1.json").write_text(json.dumps(dqr, indent=1, ensure_ascii=False, default=str))
print("===== RESUMEN (02) =====")
print(json.dumps(dqr, ensure_ascii=False, default=str, separators=(",", ":")))

===== RESUMEN (02) =====
{"entorno":"colab","n_correos":94566,"errores_parseo":{},"Q1_clase_x_grupo":{"legitimo":{"CEAS_08":17312,"Enron":15791,"Ling":2401,"Nazario":0,"Nigerian_Fraud":0,"SpamAssasin":4091,"nazario_anual":0,"phishing_pot":0,"All":39595},"phishing":{"CEAS_08":0,"Enron":0,"Ling":0,"Nazario":1565,"Nigerian_Fraud":3332,"SpamAssasin":0,"nazario_anual":3466,"phishing_pot":8614,"All":16977},"spam":{"CEAS_08":21842,"Enron":13976,"Ling":458,"Nazario":0,"Nigerian_Fraud":0,"SpamAssasin":1718,"nazario_anual":0,"phishing_pot":0,"All":37994},"All":{"CEAS_08":39154,"Enron":29767,"Ling":2859,"Nazario":1565,"Nigerian_Fraud":3332,"SpamAssasin":5809,"nazario_anual":3466,"phishing_pot":8614,"All":94566}},"Q2_duplicados":{"duplicados_intra_grupo":{"CEAS_08":191,"Enron":344,"Ling":0,"Nazario":13,"Nigerian_Fraud":41,"SpamAssasin":7,"nazario_anual":340,"phishing_pot":1766},"hashes_en_varios_grupos":441,"pares_de_grupos_con_cruce":{"Nazario + nazario_anual":439,"Enron + SpamAssasin":2},"hashes